In [1]:
import pandas as pd
import numpy as np
import glob
import os

Uploading the upward generated timeseries:

In [2]:
upward =pd.read_csv("Data/upward.csv")

I look at the initial price at time step 0 and the final price of the stock to calculate the optimal portfolio value at final timestep and the optimal allocation for each timestep: 

In [27]:
upward.loc[upward['tic'] == 'STOCK'].iloc[:991]

,date,tic,close
1000,2023-01-01,STOCK,100
1001,2023-01-02,STOCK,101
1002,2023-01-03,STOCK,102
1003,2023-01-04,STOCK,103
1004,2023-01-05,STOCK,104
...,...,...,...
1986,2025-09-13,STOCK,1086
1987,2025-09-14,STOCK,1087
1988,2025-09-15,STOCK,1088
1989,2025-09-16,STOCK,1089


The stock price starts from 101. I exclude the first price of 100 because the first initialization is fixed to 0.5, 0.5 (and then the porftolio value of day 1 considering the environments.py logics becomes fixed to 1005). So we can look at the stock price (101) from the second timestep to see what could be the maximum achievable portfolio value starting from 1005. So given that the stock price starts at 101 and ends up at 1090, the optimal portfolio value would be:

In [4]:
C = 1005
weight = 1
R = 1090/101
v = C * (weight * R + (1-weight)* 1)
v

10846.039603960397

I check that this final optimal portfolio value is the same for a run that I know it was the optimal allocation sequence and that the allocation weights are always 0,1 except for the first timestep, so that I can use this to as the optimal baseline to calculate the metrics later:

In [5]:
upward_run_work = pd.read_csv("Results/4_1/ddpg/upward/training_logs/03.csv")

Taking the first 990 days to make it comparable with experiment 3 (but episode 500 instead of 50):

In [6]:
upward_run_work.loc[(upward_run_work['episode'] == 500)].iloc[:990]

,episode,day,actions,allocation_weights,portfolio_return,reward,new_portfolio_value
498501,500,1,"0.50, 0.50","0.50, 0.50",0.50%,1005.00,"1,005"
498502,500,2,"0.00, 1.00","0.00, 1.00",0.99%,1014.95,"1,015"
498503,500,3,"0.00, 1.00","0.00, 1.00",0.98%,1024.90,"1,025"
498504,500,4,"0.00, 1.00","0.00, 1.00",0.97%,1034.85,"1,035"
498505,500,5,"0.00, 1.00","0.00, 1.00",0.96%,1044.80,"1,045"
...,...,...,...,...,...,...,...
499486,500,986,"0.00, 1.00","0.00, 1.00",0.09%,10806.24,"10,806"
499487,500,987,"0.00, 1.00","0.00, 1.00",0.09%,10816.19,"10,816"
499488,500,988,"0.00, 1.00","0.00, 1.00",0.09%,10826.14,"10,826"
499489,500,989,"0.00, 1.00","0.00, 1.00",0.09%,10836.09,"10,836"


In [7]:
optimal_upward_df = upward_run_work.loc[(upward_run_work['episode'] == 500)].iloc[:990].loc[:, ['day', 'allocation_weights', 'new_portfolio_value']]

In [8]:
assert (optimal_upward_df['allocation_weights'].iloc[1:] == "0.00, 1.00").all()

In [9]:
optimal_upward_df['new_portfolio_value'].iloc[-1] # final portfolio value as above

'10,846'

For the downward trend, it is trivial that the optimal portfolio value stays 1000:

In [10]:
C = 1000
weight = 0
R = 999/9
v = C * (weight * R + (1-weight)* 1)
v

1000.0

In [11]:
downward = pd.read_csv("Data/downward.csv")

In [12]:
downward.loc[downward['tic'] == 'STOCK'].iloc[1:992] #da 999 a 9

,date,tic,close
1001,2023-01-02,STOCK,999
1002,2023-01-03,STOCK,998
1003,2023-01-04,STOCK,997
1004,2023-01-05,STOCK,996
1005,2023-01-06,STOCK,995
...,...,...,...
1987,2025-09-14,STOCK,13
1988,2025-09-15,STOCK,12
1989,2025-09-16,STOCK,11
1990,2025-09-17,STOCK,10


I construct the optimal downward df baseline as well (to be used in the metrics calculation):

In [13]:
downward_run_work = pd.read_csv("Results/4_1/ddpg/downward/training_logs/07.csv")

Taking the first 990 days to make it comparable with experiment 3 (but episode 500 instead of 50):

In [14]:
optimal_downward_df = downward_run_work.loc[(downward_run_work['episode'] == 500)].iloc[:990].loc[:, ['day', 'allocation_weights', 'new_portfolio_value']]

In [15]:
optimal_downward_df

,day,allocation_weights,new_portfolio_value
498501,1,"0.50, 0.50","1,000"
498502,2,"1.00, 0.00","1,000"
498503,3,"1.00, 0.00","1,000"
498504,4,"1.00, 0.00","1,000"
498505,5,"1.00, 0.00","1,000"
...,...,...,...
499486,986,"1.00, 0.00","1,000"
499487,987,"1.00, 0.00","1,000"
499488,988,"1.00, 0.00","1,000"
499489,989,"1.00, 0.00","1,000"


In [16]:
periodic_df = pd.read_csv("Data/periodic.csv")

In [17]:
periodic_df.loc[periodic_df['tic'] =='STOCK'].iloc[:5]

,date,tic,close
1000,2023-01-01,STOCK,500.000000
1001,2023-01-02,STOCK,500.000000
1002,2023-01-03,STOCK,507.471907
1003,2023-01-04,STOCK,514.776010
1004,2023-01-05,STOCK,521.748277


Here I mimic the logic of what would be written in the training_logs csv files if the optimal allocation would be chosen, to obtain the optimal baseline df for the periodic trend generator: (look at timesteps_debug_periodic_alloc.txt to understand)

In [ ]:
class PeriodicTrendPriceGenerator:
    def __init__(self, start=500, amplitude=50, frequency=0.15):
        self.start = start
        self.amplitude = amplitude
        self.frequency = frequency
        self.t = 0

    def generate_price(self, last_price=None):
        value = self.amplitude * np.sin(self.frequency * self.t) + self.start
        self.t += 1
        return value

# Generate 991 days of stock prices to make it comparable with experiment 3:
days = 990 
generator = PeriodicTrendPriceGenerator(start= 500, amplitude=50, frequency=0.15)
stock_prices = [generator.generate_price() for _ in range(days)]
stock_prices = [500] + stock_prices
stock_prices = np.array(stock_prices)
allocation = [0.5,0.5]
portfolio_value = 1000
days_df = []
optimal_allocations = []
optimal_portfolio_values = []

for day, stock_price in enumerate(stock_prices):
    print("day: ", day)
    if day == 0:
       pass
    else:
        if day == 1:
           
            pass
        old_allocation = allocation
        
        print("stock price today", stock_prices[day])
        print("stock price yesterday", stock_prices[day-1])
        portfolio_return = sum((np.array([500, stock_prices[day]]) / np.array([500, stock_prices[day-1]])-1)*np.array(old_allocation))
        new_portfolio_value = portfolio_value*(1+portfolio_return)
        portfolio_value = new_portfolio_value
        print("portfolio_value", portfolio_value)
        if (stock_prices[day] >= stock_prices[day-1]):
            allocation = [0, 1]
        else:
            allocation = [1, 0]
        print("old_allocation", old_allocation)
        print("allocation:", allocation)
        days_df.append(day)
        optimal_allocations.append(old_allocation)
        optimal_portfolio_values.append(portfolio_value)

optimal_periodic_df = pd.DataFrame({"day": days_df,
                                            "allocation_weights":optimal_allocations,
                                            "new_portfolio_value": optimal_portfolio_values
                                           })


day:  0
day:  1
stock price today 500.0
stock price yesterday 500.0
portfolio_value 1000.0
old_allocation [0.5, 0.5]
allocation: [0, 1]
day:  2
stock price today 507.47190662367996
stock price yesterday 500.0
portfolio_value 1014.9438132473598
old_allocation [0, 1]
allocation: [0, 1]
day:  3
stock price today 514.776010333067
stock price yesterday 507.47190662367996
portfolio_value 1029.552020666134
old_allocation [0, 1]
allocation: [0, 1]
day:  4
stock price today 521.7482767055615
stock price yesterday 514.776010333067
portfolio_value 1043.496553411123
old_allocation [0, 1]
allocation: [0, 1]
day:  5
stock price today 528.2321236697518
stock price yesterday 521.7482767055615
portfolio_value 1056.4642473395036
old_allocation [0, 1]
allocation: [0, 1]
day:  6
stock price today 534.0819380011667
stock price yesterday 528.2321236697518
portfolio_value 1068.1638760023336
old_allocation [0, 1]
allocation: [0, 1]
day:  7
stock price today 539.1663454813742
stock price yesterday 534.08193800

In [19]:
optimal_periodic_df

,day,allocation_weights,new_portfolio_value
0,1,"[0.5, 0.5]",1000.000000
1,2,"[0, 1]",1014.943813
2,3,"[0, 1]",1029.552021
3,4,"[0, 1]",1043.496553
4,5,"[0, 1]",1056.464247
...,...,...,...
985,986,"[1, 0]",105289.941146
986,987,"[1, 0]",105289.941146
987,988,"[1, 0]",105289.941146
988,989,"[1, 0]",105289.941146


I clean the allocation weights column so that is always a list of two numbers, and the portfolio is always a float with two decimal digits:

In [20]:
def clean_allocation_df(df):
    allocation_column = df.filter(regex = 'allocation').columns[0]
    portfolio_column = df.filter(regex = 'portfolio').columns[0]
    # Convert allocation_weights to [1, 0] or [0, 1] format
    # Convert allocation_weights to list of two floats with two decimal digits
    if df[allocation_column].dtype == object:
        df[allocation_column] = df[allocation_column].apply(
            lambda x: [round(float(i.strip()), 2) for i in str(x).strip("[]").split(",")]
        )


    # Clean and convert new_portfolio_value to float with 2 decimals
    df[portfolio_column] = (
        df[portfolio_column]
        .astype(str)                      # ensure string for cleanup
        .str.replace(",", "", regex=False)  # remove thousand separators
        .astype(float)
        .round(2)
    )

    return df

optimal_upward_df = clean_allocation_df(optimal_upward_df)
optimal_periodic_df = clean_allocation_df(optimal_periodic_df)
optimal_downward_df = clean_allocation_df(optimal_downward_df)

In [21]:
optimal_upward_df.head()

,day,allocation_weights,new_portfolio_value
498501,1,"[0.5, 0.5]",1005.0
498502,2,"[0.0, 1.0]",1015.0
498503,3,"[0.0, 1.0]",1025.0
498504,4,"[0.0, 1.0]",1035.0
498505,5,"[0.0, 1.0]",1045.0


In [22]:
optimal_downward_df.head()

,day,allocation_weights,new_portfolio_value
498501,1,"[0.5, 0.5]",1000.0
498502,2,"[1.0, 0.0]",1000.0
498503,3,"[1.0, 0.0]",1000.0
498504,4,"[1.0, 0.0]",1000.0
498505,5,"[1.0, 0.0]",1000.0


In [23]:
optimal_periodic_df.head()

,day,allocation_weights,new_portfolio_value
0,1,"[0.5, 0.5]",1000.00
1,2,"[0.0, 1.0]",1014.94
2,3,"[0.0, 1.0]",1029.55
3,4,"[0.0, 1.0]",1043.50
4,5,"[0.0, 1.0]",1056.46


Now I can compute the Mean allocation error and the difference of the final portfolio value between each single run and the optimal baseline (according to the right generator associated to the run):

In [24]:
base_path = "Results/4_1"
base_path_noisy = "Results/4_2"
agents = ["a2c", "ddpg"]
regimes = ["upward", "downward", "periodic", "upward_noise", "downward_noise", "periodic_noise"]

# Result containers
result_dfs = {}  # structure: result_dfs[agent][regime]
all_data = {}

# Function to compute mean L2 error
def compute_mean_l2_error(agent_allocs, opt_allocs):
    errors = np.linalg.norm(agent_allocs - opt_allocs, axis=1)
    return np.mean(errors)

# You must define or load the optimal allocation DataFrames for each regime
optimal_data = {
    "upward": clean_allocation_df(optimal_upward_df),
    "downward": clean_allocation_df(optimal_downward_df),
    "periodic": clean_allocation_df(optimal_periodic_df),
     "upward_noise": clean_allocation_df(optimal_upward_df),
    "downward_noise": clean_allocation_df(optimal_downward_df),
    "periodic_noise": clean_allocation_df(optimal_periodic_df),
}

for agent in agents:
    result_dfs[agent] = {}
    all_data[agent] = {}
    
    for regime in regimes:
        # Prepare optimal allocations
        opt_allocs_array = np.array(optimal_data[regime]["allocation_weights"].tolist())

        # Gather CSVs
        # Select correct base path based on regime type
        current_base = base_path_noisy if "noise" in regime else base_path
        csv_pattern = os.path.join(current_base, agent, regime, "training_logs", "*.csv")

        csv_files = sorted(glob.glob(csv_pattern))
        print(csv_files)
        mean_errors = []
        dfs = []
        delta_vs = []

        for fpath in csv_files:
            df = pd.read_csv(fpath)
            df = clean_allocation_df(df.loc[df['episode'] == 50, ['day',"allocation_weights", 'new_portfolio_value']])
            agent_allocs_array = np.array(df["allocation_weights"].tolist())
            min_len = min(len(agent_allocs_array), len(opt_allocs_array))
            opt_df = optimal_data[regime]
            if (agent == 'a2c') and (regime == "downward_noise"):
                agent_allocs_array_check = agent_allocs_array
                opt_allocs_array_check = opt_allocs_array

            error = compute_mean_l2_error(agent_allocs_array[:min_len], opt_allocs_array[:min_len])
            mean_errors.append(error)
            dfs.append(df)
              # Compute ∆V
            try:
                V_agent_T = float(str(df["new_portfolio_value"].iloc[-1]).replace(",", ""))
                V_opt_T = float(str(opt_df["new_portfolio_value"].iloc[min_len - 1]).replace(",", ""))
                delta_v = V_agent_T - V_opt_T
            except Exception as e:
                delta_v = np.nan  # fallback if any issue
            delta_vs.append(delta_v)

        result_dfs[agent][regime] = pd.DataFrame({
            "run": list(range(1, len(mean_errors) + 1)),
            "mean_allocation_error": mean_errors,
            "delta_V": delta_vs
        })
        all_data[agent][regime] = pd.concat(dfs, ignore_index=True)


['Results/4_1/a2c/upward/training_logs/01.csv', 'Results/4_1/a2c/upward/training_logs/02.csv', 'Results/4_1/a2c/upward/training_logs/03.csv', 'Results/4_1/a2c/upward/training_logs/04.csv', 'Results/4_1/a2c/upward/training_logs/05.csv', 'Results/4_1/a2c/upward/training_logs/06.csv', 'Results/4_1/a2c/upward/training_logs/07.csv', 'Results/4_1/a2c/upward/training_logs/08.csv', 'Results/4_1/a2c/upward/training_logs/09.csv', 'Results/4_1/a2c/upward/training_logs/10.csv']
['Results/4_1/a2c/downward/training_logs/01.csv', 'Results/4_1/a2c/downward/training_logs/02.csv', 'Results/4_1/a2c/downward/training_logs/03.csv', 'Results/4_1/a2c/downward/training_logs/04.csv', 'Results/4_1/a2c/downward/training_logs/05.csv', 'Results/4_1/a2c/downward/training_logs/06.csv', 'Results/4_1/a2c/downward/training_logs/07.csv', 'Results/4_1/a2c/downward/training_logs/08.csv', 'Results/4_1/a2c/downward/training_logs/09.csv', 'Results/4_1/a2c/downward/training_logs/10.csv']
['Results/4_1/a2c/periodic/training_lo

This is to access the mean allocation error and the delta V for each run after choosing an agent and a generator:

In [25]:
result_dfs['a2c']['periodic']

,run,mean_allocation_error,delta_V
0,1,0.709964,-104378.94
1,2,0.710149,-104383.94
2,3,0.710278,-104377.94
3,4,0.707192,-104363.94
4,5,0.709921,-104377.94
5,6,0.710507,-104379.94
6,7,0.709935,-104377.94
7,8,0.708592,-104372.94
8,9,0.708107,-104369.94
9,10,0.712107,-104387.94


Generate the LATEX table:

In [28]:
for agent in ["a2c", "ddpg"]:
    print("\\begin{table}[ht]")
    print("  \\centering")
    print(f"  \\caption{{Expt. 1\&2 - {agent.upper()} Results Averaged Across 10 Runs with Different Generators}}")
    print(f"  \\label{{tab:{agent}_results}}")

    # --- Clean Regimes Subtable ---
    print("  \\begin{subtable}[t]{0.48\\textwidth}")
    print("  \\centering")
    print("  \\caption{Generators with no Noise}")
    print("  \\begin{tabular}{ccc}")
    print("  \\toprule")
    print("  Generator & Mean Allocation Error & $\\Delta V$ \\\\")
    print("  \\midrule")
    for regime in ["upward", "downward", "periodic"]:
        df = result_dfs[agent].get(regime)
        if df is not None and not df.empty:
            mean_error = df["mean_allocation_error"].mean() # to calculate the average Mean Allocation Error across runs
            delta_v = df["delta_V"].mean() # to calculate the average difference in final portfolio value across runs 
            print(f"  {regime.capitalize()} & {mean_error:.2f} & {delta_v:.2f} \\\\")
    print("  \\bottomrule")
    print("  \\end{tabular}")
    print("  \\end{subtable}")

    print("  \\hfill")

    # --- Noisy Regimes Subtable ---
    print("  \\begin{subtable}[t]{0.48\\textwidth}")
    print("  \\centering")
    print("  \\caption{Generators with Noise}")
    print("  \\begin{tabular}{ccc}")
    print("  \\toprule")
    print("  Generator & Mean Allocation Error & $\\Delta V$ \\\\")
    print("  \\midrule")
    for regime in ["upward_noise", "downward_noise", "periodic_noise"]:
        df = result_dfs[agent].get(regime)
        if df is not None and not df.empty:
            mean_error = df["mean_allocation_error"].mean()
            delta_v = df["delta_V"].mean()
            print(f"  {regime.replace('_noise','').capitalize()} (Noise) & {mean_error:.2f} & {delta_v:.2f} \\\\")
    print("  \\bottomrule")
    print("  \\end{tabular}")
    print("  \\end{subtable}")

    print("\\end{table}")
    print("\n\n")

\begin{table}[ht]
  \centering
  \caption{Expt. 1\&2 - A2C Results Averaged Across 10 Runs with Different Generators}
  \label{tab:a2c_results}
  \begin{subtable}[t]{0.48\textwidth}
  \centering
  \caption{Generators with no Noise}
  \begin{tabular}{ccc}
  \toprule
  Generator & Mean Allocation Error & $\Delta V$ \\
  \midrule
  Upward & 0.45 & -5395.00 \\
  Downward & 0.84 & -971.40 \\
  Periodic & 0.71 & -104377.14 \\
  \bottomrule
  \end{tabular}
  \end{subtable}
  \hfill
  \begin{subtable}[t]{0.48\textwidth}
  \centering
  \caption{Generators with Noise}
  \begin{tabular}{ccc}
  \toprule
  Generator & Mean Allocation Error & $\Delta V$ \\
  \midrule
  Upward (Noise) & 0.03 & -7072.70 \\
  Downward (Noise) & 1.40 & -464.00 \\
  Periodic (Noise) & 0.71 & -104422.04 \\
  \bottomrule
  \end{tabular}
  \end{subtable}
\end{table}



\begin{table}[ht]
  \centering
  \caption{Expt. 1\&2 - DDPG Results Averaged Across 10 Runs with Different Generators}
  \label{tab:ddpg_results}
  \begin{su